In [4]:
import time
import pandas as pd
from typing import TypedDict, Annotated, List, Dict
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import pipeline
from langgraph.graph import StateGraph, END

### LangGraph = LLM workflow engine built on top of LangChain

#### AgentState
    * Every node receives a state object
    * Every node returns a partial state update

    This increases the Tracebility and Monitorable

In [6]:
class AgentState(TypedDict):
    input: str
    output: str
    session_id: str
    request_id: str
    path: str
    history: List[str]

                Initial State
                {
                input: "Who is Elon Musk?",
                session_id: "abc",
                request_id: "123"
                }
                        ↓
                Node: reason()
                        ↓
                Updated State
                {
                output: "...",
                path: "tool_wikipedia"
                }
                        ↓
                END

#### The reason() Node

        This is your single decision node.

#### Structured Logging

In [ ]:
logger.info(
    f"Agent Step: {path}",
    extra={...}
)

* This is not debug logging.
* This is observability logging.
* You are capturing:

                | Field         | Why                    |

        | ------------- | ---------------------- |
        
        | request_id    | Trace correlation      |
        | path_selected | Behavior analysis      |
        | latency_sec   | Performance monitoring |
        | query         | Context debugging      |


## Graph Construction Explained

In [ ]:
# State type → AgentState
graph = StateGraph(AgentState) 

# Add a node.
graph.add_node("reason", reason)

# Execution starts here.
graph.set_entry_point("reason")

#After this node, terminate.
graph.add_edge("reason", END)


START → reason → END

# Implementation

        User Input
        ↓
        Router Node (reason)
        ↓
        Wikipedia OR Calculator OR LLM
        ↓
        Return Output

In [37]:
# =========================================
# LangGraph Agent using OpenAI
# =========================================

from typing import TypedDict
import time
import os

# LangChain LLM
from langchain_openai import ChatOpenAI

# Tools
from langchain.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_experimental.tools import PythonREPLTool

# LangGraph
from langgraph.graph import StateGraph, END


# ------------------------------
# 1️⃣ Set your OpenAI Key
# ------------------------------
# Make sure you export:
# export OPENAI_API_KEY="your_key_here"

llm = ChatOpenAI(
    model="gpt-4o-mini",   # you can change model
    temperature=0
)

### Create Tools

In [38]:
wiki = WikipediaAPIWrapper()
python_repl = PythonREPLTool()

@tool
def wikipedia_search(query: str) -> str:
    """Search and summarize information from Wikipedia."""
    return wiki.run(query)

@tool
def calculator(expression: str) -> str:
    """Perform arithmetic calculations."""
    return python_repl.run(expression)


In [39]:
# ------------------------------
# Agent State
# ------------------------------

class AgentState(TypedDict):
    input: str
    output: str
    path: str


# ------------------------------
# Create LangGraph Agent
# ------------------------------

def create_agent():

    def reason(state: AgentState):
        query = state["input"].lower()
        start = time.perf_counter()

        path = "llm"
        output = ""

        # Simple deterministic routing
        if "calculate" in query or any(op in query for op in ["+", "-", "*", "/"]):
            path = "calculator"
            expression = query.replace("calculate", "").strip()
            output = calculator.invoke(expression)

        elif "who" in query or "when" in query or "search" in query:
            path = "wikipedia"
            output = wikipedia_search.invoke(query)

        else:
            path = "llm"
            response = llm.invoke(query)
            output = response.content

        latency = time.perf_counter() - start
        print(f"[DEBUG] Path: {path} | Latency: {latency:.4f}s")

        return {
            "output": output.strip(),
            "path": path
        }

    graph = StateGraph(AgentState)
    graph.add_node("reason", reason)
    graph.set_entry_point("reason")
    graph.add_edge("reason", END)

    return graph.compile()


In [40]:
if __name__ == "__main__":
    agent = create_agent()

    queries = [
        "Who is Elon Musk?",
        "Calculate 45 * 3",
        "Explain quantum computing"
    ]

    for q in queries:
        result = agent.invoke({"input": q})
        print("\nQuery:", q)
        print("Path:", result["path"])
        print("Response:", result["output"])

[DEBUG] Path: wikipedia | Latency: 5.0086s

Query: Who is Elon Musk?
Path: wikipedia
Response: Page: Errol Musk
Summary: Errol Graham Musk (born 25 May 1946) is a South African businessman, politician, and the patriarch of the Musk family. He was an Independent member of the Pretoria City Council from 1972 to 1980, and then as a member of Progressive Federal Party from 1980 to 1983. As a businessman, he has worked as a mechanical engineering consultant, developed properties, and invested in various ventures including emerald trading.
Musk married Maye Haldeman in 1970, and they had three children: Elon, Kimbal and Tosca. Haldeman divorced him in 1979, saying he was abusive. He has since had two other marriages and four other children. His son Elon is the wealthiest person in the world.

Page: Elon Musk
Summary: Elon Reeve Musk ( EE-lon; born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the

## Routing as Node

In [20]:
# =========================================
# LangGraph with Explicit Router Node
# =========================================

from typing import TypedDict
import time

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_experimental.tools import PythonREPLTool

from langgraph.graph import StateGraph, END

# ------------------------------
# 1️⃣ LLM Setup
# ------------------------------

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ------------------------------
# 2️⃣ Tools
# ------------------------------

wiki = WikipediaAPIWrapper()
python_repl = PythonREPLTool()

@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return a concise summary."""
    return wiki.run(query)


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return python_repl.run(expression)

# ------------------------------
# 3️⃣ Agent State
# ------------------------------

class AgentState(TypedDict):
    input: str
    output: str
    route: str

# ------------------------------
# 4️⃣ Router Node
# ------------------------------

def router_node(state: AgentState):
    query = state["input"].lower()

    if "calculate" in query or any(op in query for op in ["+", "-", "*", "/"]):
        route = "calculator"
    elif "who" in query or "when" in query or "search" in query:
        route = "wikipedia"
    else:
        route = "llm"

    print(f"[ROUTER] Selected route: {route}")

    return {"route": route}

# ------------------------------
# 5️⃣ Tool Nodes
# ------------------------------

def calculator_node(state: AgentState):
    query = state["input"].lower().replace("calculate", "").strip()
    result = calculator.invoke(query)
    return {"output": result}

def wikipedia_node(state: AgentState):
    result = wikipedia_search.invoke(state["input"])
    return {"output": result}

def llm_node(state: AgentState):
    response = llm.invoke(state["input"])
    return {"output": response.content}

# ------------------------------
# 6️⃣ Build Graph
# ------------------------------

def create_agent():

    graph = StateGraph(AgentState)

    graph.add_node("router", router_node)
    graph.add_node("calculator", calculator_node)
    graph.add_node("wikipedia", wikipedia_node)
    graph.add_node("llm", llm_node)

    graph.set_entry_point("router")

    # Conditional routing
    graph.add_conditional_edges(
        "router",
        lambda state: state["route"],
        {
            "calculator": "calculator",
            "wikipedia": "wikipedia",
            "llm": "llm"
        }
    )

    # All tool nodes end
    graph.add_edge("calculator", END)
    graph.add_edge("wikipedia", END)
    graph.add_edge("llm", END)

    return graph.compile()

# ------------------------------
# 7️⃣ Run
# ------------------------------

if __name__ == "__main__":
    agent = create_agent()

    queries = [
        "Who is Elon Musk?",
        "Calculate 10 * 5",
        "Explain blockchain"
    ]

    for q in queries:
        result = agent.invoke({"input": q})
        print("\nQuery:", q)
        print("Response:", result["output"])

[ROUTER] Selected route: wikipedia

Query: Who is Elon Musk?
Response: Page: Errol Musk
Summary: Errol Graham Musk (born 25 May 1946) is a South African businessman, politician, and the patriarch of the Musk family. He was an Independent member of the Pretoria City Council from 1972 to 1980, and then as a member of Progressive Federal Party from 1980 to 1983. As a businessman, he has worked as a mechanical engineering consultant, developed properties, and invested in various ventures including emerald trading.
Musk married Maye Haldeman in 1970, and they had three children: Elon, Kimbal and Tosca. Haldeman divorced him in 1979, saying he was abusive. He has since had two other marriages and four other children. His son Elon is the wealthiest person in the world.

Page: Elon Musk
Summary: Elon Reeve Musk ( EE-lon; born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the world since 2025; as of

In [25]:
mermaid = agent.get_graph().draw_mermaid()
print(mermaid)

# https://mermaid.live

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	calculator(calculator)
	wikipedia(wikipedia)
	llm(llm)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> calculator;
	router -.-> llm;
	router -.-> wikipedia;
	calculator --> __end__;
	llm --> __end__;
	wikipedia --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Resume Analyzer

        create_react_agent() (prebuilt black-box agent) to Custom LangGraph orchestration (explicit graph control)

        agent = create_react_agent(...)

This hides:
        * Planning
        * Tool selection
        * Execution
        * Final response

In [27]:
import json
import re
from typing import TypedDict, Optional

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import StateGraph, END

## LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def clean_json(text: str) -> dict:
    cleaned = re.sub(r"```json|```", "", text).strip()
    return json.loads(cleaned)


## Tools 

@tool
def analyze_experience(user_profile: str) -> dict:
    """Extract experience and seniority from a user profile."""
    prompt = f"""
    Analyze the user profile and extract experience information.

    USER PROFILE:
    {user_profile}

    Respond ONLY in JSON.
    """
    response = llm.invoke(prompt)
    return clean_json(response.content)


@tool
def analyze_competencies(user_profile: str) -> dict:
    """Identify competencies."""
    prompt = f"""
    Identify core competencies.

    USER PROFILE:
    {user_profile}

    Respond ONLY in JSON.
    """
    response = llm.invoke(prompt)
    return clean_json(response.content)


@tool
def recommend_roles(user_profile: str) -> dict:
    """Recommend suitable roles."""
    prompt = f"""
    Recommend job roles.

    USER PROFILE:
    {user_profile}

    Respond ONLY in JSON.
    """
    response = llm.invoke(prompt)
    return clean_json(response.content)


# Define Agents
class AgentState(TypedDict):
    input: str
    tool_to_call: Optional[str]
    tool_output: Optional[dict]
    final_output: Optional[str]

# Planner Node (LLM decides which tool)
def planner_node(state: AgentState):
    query = state["input"]

    prompt = f"""
    You are a routing agent.

    Choose one tool:
    - analyze_experience
    - analyze_competencies
    - recommend_roles

    USER PROFILE:
    {query}

    Return ONLY the tool name.
    """

    response = llm.invoke(prompt)
    tool_name = response.content.strip()

    print(f"[Planner] Selected tool: {tool_name}")

    return {"tool_to_call": tool_name}

# Tool Execution Node

def tool_node(state: AgentState):
    tool_name = state["tool_to_call"]
    profile = state["input"]

    tool_map = {
        "analyze_experience": analyze_experience,
        "analyze_competencies": analyze_competencies,
        "recommend_roles": recommend_roles
    }

    selected_tool = tool_map.get(tool_name)

    if not selected_tool:
        return {"tool_output": {"error": "Invalid tool selected"}}

    result = selected_tool.invoke(profile)

    return {"tool_output": result}

def final_node(state: AgentState):
    output = state["tool_output"]
    return {"final_output": json.dumps(output, indent=2)}


## Builde Langgraph
def create_agent():

    graph = StateGraph(AgentState)

    graph.add_node("planner", planner_node)
    graph.add_node("tool_executor", tool_node)
    graph.add_node("final", final_node)

    graph.set_entry_point("planner")

    graph.add_edge("planner", "tool_executor")
    graph.add_edge("tool_executor", "final")
    graph.add_edge("final", END)

    return graph.compile()

      START
         ↓
      planner
         ↓
      tool_executor
         ↓
      final
         ↓
      END

In [29]:
if __name__ == "__main__":

    agent = create_agent()

    profile = """
    8 years experience in AI and Machine Learning.
    Strong in Python, NLP, LangChain, and LLM systems.
    """

    result = agent.invoke({"input": profile})

    print(result["final_output"])

[Planner] Selected tool: recommend_roles
{
  "recommended_job_roles": [
    {
      "title": "Senior Machine Learning Engineer",
      "description": "Lead the design and implementation of machine learning models and algorithms, focusing on NLP and LLM systems."
    },
    {
      "title": "AI Research Scientist",
      "description": "Conduct cutting-edge research in artificial intelligence, particularly in natural language processing and large language models."
    },
    {
      "title": "NLP Engineer",
      "description": "Develop and optimize NLP applications, leveraging expertise in Python and LangChain to enhance language understanding."
    },
    {
      "title": "Data Scientist - AI Focus",
      "description": "Analyze complex datasets to derive insights and build predictive models using machine learning techniques."
    },
    {
      "title": "Machine Learning Architect",
      "description": "Design scalable machine learning architectures and frameworks, ensuring efficie

In [31]:
mermaid = agent.get_graph().draw_mermaid()
print(mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	tool_executor(tool_executor)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	planner --> tool_executor;
	tool_executor --> final;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



        START
        ↓
        planner
        ↓
        tool_executor
        ↓
        reflection
        ↓
        guardrails
        ↓
        cost_monitor
        ↓
        END

In [34]:
# STEP 1 — Upgrade State Structure

from typing import TypedDict, List, Optional, Dict

class AgentState(TypedDict):
    input: str
    planned_tools: List[str]
    current_tool: Optional[str]
    tool_outputs: List[Dict]
    final_output: Optional[str]
    retry_count: int
    total_tokens: int
    total_latency: float
    validation_passed: bool

# Step 2 - Multi Step Planner

def planner_node(state: AgentState):

    prompt = f"""
    Decide which tools to call in sequence.
    Available tools:
    - analyze_experience
    - analyze_competencies
    - recommend_roles

    Return a JSON list of tool names.
    """

    response = llm.invoke(prompt)

    tools = json.loads(response.content)

    return {
        "planned_tools": tools,
        "retry_count": 0
    }

#STEP 3 — Conditional Tool Executor (Loop Support)
def tool_executor_node(state: AgentState):

    if not state["planned_tools"]:
        return {}

    tool_name = state["planned_tools"][0]
    remaining = state["planned_tools"][1:]

    tool_map = {
        "analyze_experience": analyze_experience,
        "analyze_competencies": analyze_competencies,
        "recommend_roles": recommend_roles
    }

    start = time.perf_counter()

    result = tool_map[tool_name].invoke(state["input"])

    latency = time.perf_counter() - start

    return {
        "current_tool": tool_name,
        "planned_tools": remaining,
        "tool_outputs": state.get("tool_outputs", []) + [result],
        "total_latency": state.get("total_latency", 0) + latency
    }


# STEP 4 — Add Exponential Backoff Retry

import time

def retry_wrapper(tool_fn, input_text, retry_count):
    try:
        return tool_fn.invoke(input_text)
    except Exception:
        if retry_count >= 3:
            raise
        wait = 2 ** retry_count
        time.sleep(wait)
        return retry_wrapper(tool_fn, input_text, retry_count + 1)


# STEP 5 — Reflection Node

# def reflection_node(state: AgentState):

#     summary_prompt = f"""
#     Based on these outputs:
#     {state["tool_outputs"]}

#     Provide final structured recommendation.
#     """

#     response = llm.invoke(summary_prompt)

#     return {"final_output": response.content}

def reflection_node(state: AgentState):
    print("\n[Reflection Input State]")
    print(state)

    response = llm.invoke(...)

    print("\n[Reflection Output]")
    print(response.content)

    return {"final_output": response.content}
# This simulates chain-of-thought refinement.

# STEP 6 — Guardrails Node (Validation)
## Basic Schema
# def guardrails_node(state: AgentState):

#     try:
#         json.loads(state["final_output"])
#         return {"validation_passed": True}
#     except:
#         return {"validation_passed": False}

## Actual
ALLOWED_ROLES = [
    "AI Architect",
    "Principal Data Scientist",
    "ML Engineering Lead",
    "Senior Data Scientist",
    "AI Consultant"
]

def guardrails_node(state: AgentState):

    try:
        data = json.loads(state["final_output"])

        # 1️⃣ Required fields
        required = ["summary", "career_direction", "recommended_next_steps"]
        for field in required:
            if field not in data:
                return {"validation_passed": False}

        # 2️⃣ Validate roles
        if "recommended_next_steps" in data:
            for role in data["recommended_next_steps"]:
                if role not in ALLOWED_ROLES:
                    print(f"⚠ Invalid role detected: {role}")
                    return {"validation_passed": False}

        return {"validation_passed": True}

    except Exception as e:
        print("Guardrail error:", e)
        return {"validation_passed": False}



# STEP 7 — Cost & Token Tracking
def token_tracker_node(state: AgentState):
    tokens = len(state["final_output"].split())
    return {
        "total_tokens": state.get("total_tokens", 0) + tokens
    }

# STEP 8 — Conditional Routing
graph = StateGraph(AgentState)

graph.add_node("planner", planner_node)
graph.add_node("executor", tool_executor_node)
graph.add_node("reflection", reflection_node)
graph.add_node("guardrails", guardrails_node)
graph.add_node("token_tracker", token_tracker_node)

graph.set_entry_point("planner")

graph.add_edge("planner", "executor")

graph.add_conditional_edges(
    "executor",
    lambda s: "executor" if s["planned_tools"] else "reflection",
    {
        "executor": "executor",
        "reflection": "reflection"
    }
)

graph.add_edge("reflection", "guardrails")

# graph.add_conditional_edges(
#     "guardrails",
#     lambda s: "token_tracker" if s["validation_passed"] else "reflection",
#     {
#         "token_tracker": "token_tracker",
#         "reflection": "reflection"
#     }
# )

graph.add_conditional_edges(
    "guardrails",
    lambda s: "reflection" if not s["validation_passed"] else "token_tracker",
    {
        "reflection": "reflection",
        "token_tracker": "token_tracker"
    }
)

graph.add_edge("token_tracker", END)



if __name__ == "__main__":

    agent = create_agent()

    profile = """
    8 years experience in AI and Machine Learning.
    Strong in Python, NLP, LangChain, and LLM systems.
    """

    #result = agent.invoke({"input": profile})
    for step in agent.stream({"input": profile}):
        print(step)

    #print(result["final_output"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

        START
        ↓
        planner
        ↓
        executor (loop until tools exhausted)
        ↓
        reflection
        ↓
        guardrails
        ↺ (if invalid)
        ↓
        token_tracker
        ↓
        END

In [36]:
import json
import time
from typing import TypedDict, List, Optional, Dict
from langgraph.graph import StateGraph, END


from pydantic import BaseModel
from typing import List

class ToolPlan(BaseModel):
    tools: List[str]



# ------------------------------
# STATE
# ------------------------------

class AgentState(TypedDict, total=False):
    input: str
    planned_tools: List[str]
    current_tool: Optional[str]
    tool_outputs: List[Dict]
    final_output: Optional[str]
    retry_count: int
    total_tokens: int
    total_latency: float
    validation_passed: bool
    correction_attempts: int

# ------------------------------
# PLANNER
# ------------------------------

def planner_node(state: AgentState):

    structured_llm = llm.with_structured_output(ToolPlan)

    prompt = """
    Decide which tools to call in sequence.
    Available tools:
    - analyze_experience
    - analyze_competencies
    - recommend_roles
    """

    response = structured_llm.invoke(prompt)

    return {
        "planned_tools": response.tools,
        "tool_outputs": [],
        "retry_count": 0,
        "total_latency": 0,
        "total_tokens": 0,
        "correction_attempts": 0
    }

    

# def planner_node(state: AgentState):
#     prompt = """
#     Decide which tools to call in sequence.
#     Available tools:
#     - analyze_experience
#     - analyze_competencies
#     - recommend_roles

#     Return JSON list.
#     """

#     response = llm.invoke(prompt)

#     tools = json.loads(response.content)

#     return {
#         "planned_tools": tools,
#         "tool_outputs": [],
#         "retry_count": 0,
#         "total_latency": 0,
#         "total_tokens": 0,
#         "correction_attempts": 0
#     }

# ------------------------------
# RETRY WRAPPER
# ------------------------------

def retry_wrapper(tool_fn, input_text, retry_count=0):
    try:
        return tool_fn.invoke(input_text)
    except Exception:
        if retry_count >= 3:
            raise
        wait = 2 ** retry_count
        time.sleep(wait)
        return retry_wrapper(tool_fn, input_text, retry_count + 1)

# ------------------------------
# EXECUTOR
# ------------------------------

def tool_executor_node(state: AgentState):

    if not state.get("planned_tools"):
        return {}

    tool_name = state["planned_tools"][0]
    remaining = state["planned_tools"][1:]

    tool_map = {
        "analyze_experience": analyze_experience,
        "analyze_competencies": analyze_competencies,
        "recommend_roles": recommend_roles
    }

    start = time.perf_counter()

    result = retry_wrapper(tool_map[tool_name], state["input"])

    latency = time.perf_counter() - start

    return {
        "current_tool": tool_name,
        "planned_tools": remaining,
        "tool_outputs": state["tool_outputs"] + [result],
        "total_latency": state["total_latency"] + latency
    }

# ------------------------------
# REFLECTION
# ------------------------------

def reflection_node(state: AgentState):

    prompt = f"""
    Based on tool outputs:
    {state['tool_outputs']}

    Generate final structured JSON with:
    summary
    career_direction
    recommended_next_steps
    """

    response = llm.invoke(prompt)

    return {"final_output": response.content}

# ------------------------------
# GUARDRAILS
# ------------------------------

ALLOWED_ROLES = [
    "AI Architect",
    "Principal Data Scientist",
    "ML Engineering Lead",
    "Senior Data Scientist",
    "AI Consultant"
]

def guardrails_node(state: AgentState):

    attempts = state.get("correction_attempts", 0)

    try:
        data = json.loads(state["final_output"])

        required = ["summary", "career_direction", "recommended_next_steps"]
        for field in required:
            if field not in data:
                return {
                    "validation_passed": False,
                    "correction_attempts": attempts + 1
                }

        for role in data["recommended_next_steps"]:
            if role not in ALLOWED_ROLES:
                return {
                    "validation_passed": False,
                    "correction_attempts": attempts + 1
                }

        return {"validation_passed": True}

    except:
        return {
            "validation_passed": False,
            "correction_attempts": attempts + 1
        }

# ------------------------------
# TOKEN TRACKER
# ------------------------------

def token_tracker_node(state: AgentState):

    tokens = len(state["final_output"].split())

    return {
        "total_tokens": state["total_tokens"] + tokens
    }

# ------------------------------
# BUILD GRAPH
# ------------------------------

def create_agent():

    graph = StateGraph(AgentState)

    graph.add_node("planner", planner_node)
    graph.add_node("executor", tool_executor_node)
    graph.add_node("reflection", reflection_node)
    graph.add_node("guardrails", guardrails_node)
    graph.add_node("token_tracker", token_tracker_node)

    graph.set_entry_point("planner")

    graph.add_edge("planner", "executor")
    
# After executor finishes → evaluate state → choose next node.
    graph.add_conditional_edges(
        "executor",
        lambda s: "executor" if s.get("planned_tools") else "reflection",
        {
            "executor": "executor",
            "reflection": "reflection"
        }
    )

    graph.add_edge("reflection", "guardrails")

    graph.add_conditional_edges(
        "guardrails",
        lambda s: "reflection"
        if not s.get("validation_passed") and s.get("correction_attempts", 0) < 2
        else "token_tracker",
        {
            "reflection": "reflection",
            "token_tracker": "token_tracker"
        }
    )

    graph.add_edge("token_tracker", END)

    return graph.compile()

# ------------------------------
# RUN
# ------------------------------

if __name__ == "__main__":

    agent = create_agent()

    profile = """
    8 years experience in AI and Machine Learning.
    Strong in Python, NLP, LangChain, and LLM systems.
    """

    for step in agent.stream({"input": profile}):
        print(step)



{'planner': {'planned_tools': ['analyze_experience', 'analyze_competencies', 'recommend_roles'], 'tool_outputs': [], 'retry_count': 0, 'total_latency': 0, 'total_tokens': 0, 'correction_attempts': 0}}
{'executor': {'current_tool': 'analyze_experience', 'planned_tools': ['analyze_competencies', 'recommend_roles'], 'tool_outputs': [{'experience': {'years': 8, 'fields': ['AI', 'Machine Learning'], 'skills': ['Python', 'NLP', 'LangChain', 'LLM systems']}}], 'total_latency': 2.1139361249988724}}
{'executor': {'current_tool': 'analyze_competencies', 'planned_tools': ['recommend_roles'], 'tool_outputs': [{'experience': {'years': 8, 'fields': ['AI', 'Machine Learning'], 'skills': ['Python', 'NLP', 'LangChain', 'LLM systems']}}, {'core_competencies': {'technical_skills': {'programming_languages': ['Python'], 'machine_learning': ['Supervised Learning', 'Unsupervised Learning', 'Reinforcement Learning'], 'natural_language_processing': ['Text Preprocessing', 'Sentiment Analysis', 'Named Entity Rec